#  How is the performance of encoders affected by the refinement strategy?

## Define base configs

In [ ]:
import pandas as pd
metric_used = 'accuracy'
apply_correction_factor = True


def extract_metric_target(df):
    metric_map = {
        "har": "accuracy",
        "hapt": "accuracy",
    }
    return df["pipeline/task"].map(metric_map)


def extract_metric(df):
    metric_results = []
    metric_map = {
        "har": "metric/classification/accuracy",
        "hapt": "metric/classification/accuracy",
    }

    for _, row in df.iterrows():
        metric_column = metric_map[row["pipeline/task"]]
        metric_value = row[metric_column]
        metric_results.append(metric_value)

    return metric_results

In [ ]:
from pathlib import Path


from utils import (
    calculate_variant_wilcoxon,
    create_precedence_graph,
    prepare_experiment_df,
    plot_comparison_bar
)

# disable warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Path to the experiment results (parsed) summarized_executions_fixed
filename = 'clean_saved_metrics_paper_final'
summarized_executions_path = Path(f"{filename}.csv")

# summarized_executions_path = Path("summarized_executions_fixed.csv")

# Location to save figures and tables
figures_path = Path("results/figures/")
tables_path = Path("results/tables/")
figures_path.mkdir(parents=True, exist_ok=True)
tables_path.mkdir(parents=True, exist_ok=True)
print(f"Sucessfully created directories '{figures_path}' and '{tables_path}'")

## Utility functions

First, we define some utility functions that will be used to parse the benchmarks results and to generate the plots.

In [ ]:
def result_pairwise_wilcoxon(
    df, variants_variables, filters={}, show_stats=False,apply_bonferroni= False,
):
    df = prepare_experiment_df(
        df,
        **filters,
        verbose=True,
    )
    # display(df)
    if show_stats:
        print("Dataframe has", len(df.index), "rows")
        for col in df.columns:
            values = df[col].unique()
            print(f" - {col} ({len(values)})", values)

    return calculate_variant_wilcoxon(df, variants_variables, threshold=0.05,apply_bonferroni= apply_bonferroni)


def result_precedence_graph(
    df, variants_variables, filters={}, show_stdev=False,apply_bonferroni= False,show_df_tests=False
):
    w_df = result_pairwise_wilcoxon(df, variants_variables, filters,apply_bonferroni= apply_bonferroni,)
    if show_df_tests:
        print("Pairwise Wilcoxon test results:")
        display(w_df)
    return create_precedence_graph(w_df, show_stdev=show_stdev,apply_bonferroni= apply_bonferroni),w_df


def show_precedence_graph(
    df, variants_variables, filters, filename_suffix=None, show_stdev=False,apply_bonferroni= False,show_df_tests=False,filename=None
):

    dot,w_df = result_precedence_graph(
        df=df,
        variants_variables=variants_variables,
        filters=filters,
        show_stdev=show_stdev,
        apply_bonferroni=apply_bonferroni,
        show_df_tests=show_df_tests,
    )
    if filename is None:
        
        filename = "precedence_graph-" + "-".join(variants_variables)
    if filename_suffix:
        filename = filename + "-" + filename_suffix
    dot.render(
        filename=filename, directory=figures_path, format="png", cleanup=True
    )
    print(f"Precedence graph saved to '{figures_path / filename}.png'\n")
    display(dot)
    return dot,w_df


def summarize_backbone_performance(df):
    # Get all unique backbones (from both Variant 1 and Variant 2)
    all_backbones = set(df["Variant 1"]).union(set(df["Variant 2"]))
    
    # Initialize a dictionary to store stats for each backbone
    backbone_stats = {}
    
    for backbone in all_backbones:
        # Get all rows where the backbone appears (either in Variant 1 or Variant 2)
        mask = (df["Variant 1"] == backbone) | (df["Variant 2"] == backbone)
        relevant_rows = df[mask]
        
        # Extract means and stds where the backbone is involved
        means = []
        stds = []
        
        for _, row in relevant_rows.iterrows():
            if row["Variant 1"] == backbone:
                means.append(row["Variant 1 Mean"])
                stds.append(row["Variant 1 stdev"])
            else:
                means.append(row["Variant 2 Mean"])
                stds.append(row["Variant 2 stdev"])
        
        # Compute mean and std across all occurrences
        mean_performance = np.mean(means) if means else 0
        avg_std = np.mean(stds) if stds else 0
        
        backbone_stats[backbone] = {
            "Mean": mean_performance,
            "Std": avg_std,
            "Mean ± Std": f"{np.round(mean_performance*100, 1)}% ± {np.round(avg_std*100, 1)}%"
        }
    # --- Original logic for wins/losses ---
    sig_df = df[df["Significant"] == True].copy()
    
    # Determine winner and loser based on means
    sig_df["Winner"] = sig_df.apply(
        lambda row: row["Variant 1"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 2"], 
        axis=1
    )
    sig_df["Loser"] = sig_df.apply(
        lambda row: row["Variant 2"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 1"], 
        axis=1
    )
    
    # Count wins and losses
    win_counts = sig_df["Winner"].value_counts()
    loss_counts = sig_df["Loser"].value_counts()
    
    # Create summary DataFrame
    summary_df = pd.DataFrame.from_dict(backbone_stats, orient="index")
    summary_df.index.name = "Backbone"
    summary_df.reset_index(inplace=True)
    
    # Ensure all backbones are included (even if no wins/losses)
    summary_df["Wins"] = summary_df["Backbone"].map(win_counts).fillna(0).astype(int)
    summary_df["Losses"] = summary_df["Backbone"].map(loss_counts).fillna(0).astype(int)
    summary_df["Net Score"] = summary_df["Wins"] - summary_df["Losses"]
    
    # Sort by Net Score (descending)
    summary_df.sort_values(["Net Score", "Mean"], ascending=[False, False], inplace=True)
    
    return summary_df


In [ ]:
def aggregate_backbone_performance(combined_df):
    """
    Processes a combined DataFrame of backbone results to produce:
    1. Technique-specific performance (mean ± std)
    2. Backbone totals across all techniques
    
    Args:
        combined_df: DataFrame containing results from multiple techniques
        
    Returns:
        tuple: (technique_summary_df, backbone_totals_df)
    """
    # --- Technique-Specific Summary ---
    technique_summary = combined_df.copy()
    
    # Convert to percentages if needed (assuming original means are 0-1)
    technique_summary["Mean"] = technique_summary["Mean"] * 100
    technique_summary["Std"] = technique_summary["Std"] * 100
    
    # Format performance string
    technique_summary["Performance"] = (
        technique_summary["Mean"].round(1).astype(str) + 
        "% ± " + 
        technique_summary["Std"].round(1).astype(str) + "%"
    )
    
    # Sort by Net Score then Mean
    technique_summary = technique_summary.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # --- Backbone Totals ---
    # Extract backbone name (before " + ")
    combined_df["Backbone Only"] = combined_df["Backbone"].str.split(" \+ ").str[0]
    
    # Group and aggregate
    backbone_totals = combined_df.groupby("Backbone Only").agg({
        "Wins": "sum",
        "Losses": "sum",
        "Net Score": "sum",
        "Mean": lambda x: np.mean(x) * 100,  # Convert to percentage
        "Std": lambda x: np.mean(x) * 100
    }).reset_index()
    
    # Format performance
    backbone_totals["Performance"] = (
        backbone_totals["Mean"].round(1).astype(str) + 
        "% ± " + 
        backbone_totals["Std"].round(1).astype(str) + "%"
    )
    
    # Clean up
    backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
    backbone_totals = backbone_totals.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # Special handling for TS2Vec if present
    if "TS2Vec" in backbone_totals["Backbone"].values:
        backbone_totals["Backbone"] = backbone_totals["Backbone"].replace({
            "TS2Vec": "TS2Vec (Partial)"
        })
    
    # Select final columns
    backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]
    
    return technique_summary, backbone_totals

def generate_performance_tables(df_list, technique_names=None):
    """
    Complete workflow from individual technique DataFrames to final tables
    
    Args:
        df_list: List of DataFrames for each technique
        technique_names: Optional list of technique names
        
    Returns:
        tuple: (combined_df, technique_summary, backbone_totals)
    """
    # Add technique identifiers if provided
    if technique_names and len(technique_names) == len(df_list):
        for df, name in zip(df_list, technique_names):
            df["Technique"] = name
    
    # Combine all DataFrames
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Generate summary tables
    technique_summary, backbone_totals = aggregate_backbone_performance(combined_df)
    
    return combined_df, technique_summary, backbone_totals



In [ ]:
summarized_executions_path

In [ ]:
df = pd.read_csv(summarized_executions_path)
df

backbones cnn better but a lot of variance we need to take a close look on whats happening

### P2. Qual o melhor backbone geral para HAR de acordo com a estratégia de refino?

Qual o melhor backbone para HAR usando a estratégia de refino Freeze?
Qual o melhor backbone para HAR usando a estratégia de refino Finetune?


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


plot_df = df.copy()
plot_df["metric"] *= 100

# Group by backbone and pretext task
# df_grouped = plot_df.groupby(["backbone", "tsk_pretext"], as_index=False).mean()
# df_grouped = plot_df.groupby(["backbone", "tsk_pretext"], as_index=False)["metric"].mean()

# df_grouped.rename(columns={"metric": "Score"}, inplace=True)

# Calculate mean and std
df_grouped = plot_df.groupby(["backbone", "tsk_pretext"], as_index=False)["metric"].agg(["mean", "std"]).reset_index()
df_grouped.rename(columns={"mean": "Score", "std": "Error"}, inplace=True)

df_grouped = df_grouped.sort_values(by=["tsk_pretext", "Score"], ascending=[True, False])

# Custom backbone order and Viridis palette
custom_palette = {
    "RNN": "tab:blue",
    "IMU Transformer": "tab:orange",
    "ResNet-SE-5": "tab:green",
    "CNN-PFF": "tab:red",
    'TS2Vec Encoder': 'tab:purple',
    'TS-TCC Encoder': 'tab:brown',
}
# Define the order of backbones
backbone_order = ["RNN", "IMU Transformer", "ResNet-SE-5", "CNN-PFF",'TS2Vec Encoder','TS-TCC Encoder']
# Get colors from Viridis
viridis_colors = sns.color_palette("deep", n_colors=len(backbone_order))
viridis_palette = {backbone: color for backbone, color in zip(backbone_order, viridis_colors)}

# Aesthetic settings
sns.set(style="whitegrid", font_scale=1.5)
custom_params = {
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",
    "grid.color": "black",
    "grid.linestyle": "--",
    "grid.linewidth": 1.5
}
sns.set_context("paper", rc=custom_params)

label_fontsize = 18
tick_fontsize = 16

# # Plot
# plt.figure(figsize=(10, 6))
# ax = sns.barplot(
#     x="tsk_pretext",
#     y="Score",
#     hue="backbone",
#     data=df_grouped,
#     hue_order=backbone_order,
#     palette=viridis_palette,
#     ci=None,           # Disable built-in CI
#     errwidth=1.5,
#     capsize=0.05
# )

# # Manually add error bars
# for i, bar in enumerate(ax.patches):
#     # Match error bar to correct data point
#     group_index = i // len(backbone_order)
#     within_group_index = i % len(backbone_order)
#     subset = df_grouped[df_grouped["tsk_pretext"] == df_grouped["tsk_pretext"].unique()[group_index]]
#     error = subset.iloc[within_group_index]["Error"]
#     height = bar.get_height()
#     ax.errorbar(
#         bar.get_x() + bar.get_width() / 2,
#         height,
#         yerr=error,
#         ecolor='black',
#         capsize=5,
#         fmt='none'
#     )


# plt.xlabel("Pretext Task", fontsize=label_fontsize, fontweight='bold')
# plt.ylabel("Balanced Accuracy (%)", fontsize=label_fontsize, fontweight='bold')
# plt.xticks(rotation=30, fontsize=tick_fontsize, fontweight='bold')
# plt.yticks(fontsize=tick_fontsize, fontweight='bold')
# plt.ylim(0, 100)

# # Legend styling
# legend = plt.legend(
#     title="Backbone",
#     bbox_to_anchor=(0.5, 1.15),  # center and slightly above
#     loc="lower center",
#     fontsize=14,
#     ncol=len(backbone_order)    # one column per legend entry
# )
# plt.setp(legend.get_title(), fontsize=14, fontweight='bold')


# plt.tight_layout()
# plt.savefig("ssl_sl_rq2.png", dpi=300, bbox_inches="tight")
# plt.show()
# plt.close()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style parameters
sns.set_style("whitegrid", rc={
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",
    "grid.color": "#DDDDDD",
    "grid.linewidth": 1.2
})

# Aesthetic scaling
sns.set_context("paper", rc={"axes.labelsize": 16, "xtick.labelsize": 14, "ytick.labelsize": 14})

# Create figure
plt.figure(figsize=(10, 6))

# Boxplot
ax = sns.boxplot(
    x='ft_strategy', 
    y='metric', 
    hue='backbone', 
    data=plot_df, 
    hue_order=backbone_order, 
    palette=viridis_palette,
    linewidth=1.6,
    fliersize=3,
    width=0.7
)

# Labels & title
# plt.title("Impact of Fine-Tuning Strategy on Model Accuracy", 
#           fontsize=16, pad=14, fontweight='bold')
plt.xlabel("Fine-Tuning Strategy", fontsize=18, labelpad=12, fontweight='bold')
plt.ylabel("Balanced Accuracy (%)", fontsize=18, labelpad=12, fontweight='bold')

# Format ticks
rotation_angle = 30 if len(df['ft_strategy'].unique()) > 3 else 0
plt.xticks(rotation=rotation_angle, fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.ylim(0, 100)

# Legend
legend = plt.legend(
    title="Backbone",
    title_fontsize=13,
    fontsize=12,
    bbox_to_anchor=(0.5, 1.15),
    loc='lower center',
    ncol=len(backbone_order),
    frameon=True
)
plt.setp(legend.get_title(), fontweight='bold')

# Layout
plt.tight_layout()
plt.savefig("ssl_sl_rq3.png", dpi=300, bbox_inches='tight', transparent=False)
plt.show()
plt.close()


In [ ]:
# import numpy as np
# # FULL FINETUNE with TS2VEC for available techniques
# dot,df_finetune_rq3_ts2vec = show_precedence_graph(
#     df,
#     variants_variables=["backbone","ft_strategy"],                # Name of the columns to compare (variant)
#     filters={
#         # "select_frac_dtarget": [1.0],               # Only use the full dataset
#         "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
#         # "select_backbones": [ "ResNet","RNN","Transformer","CNN"],   # Only use these backbones "TS2Vec",
#         # "select_tsk_pretext": ["Supervised", "TNC","Diet","TFC","LFR"],
#     },
#     show_stdev=True,
#     apply_bonferroni= apply_correction_factor,
#     filename = "wilcoxon_finetune_rq3_with_ts2vec"
    
# )

# summary_df_finetune_rq3_ts2vec= summarize_backbone_performance(df_finetune_rq3_ts2vec)
# display(summary_df_finetune_rq3_ts2vec)

# # FREEZE with TS2VEC for available techniques

# dot,df_freeze_rq3_with_ts2vec = show_precedence_graph(
#     df,
#     variants_variables=["backbone","ft_strategy"],                # Name of the columns to compare (variant)
#     filters={
#         # "select_frac_dtarget": [1.0],               # Only use the full dataset
#         "select_ft_strategy": ["Freeze"],    # Only use the full finetuning strategy
#         # "select_backbones": [ "ResNet","RNN","Transformer","CNN"],   # Only use these backbones "TS2Vec",
#         "select_tsk_pretext": ["Supervised", "TNC","Diet","TFC","LFR"],
#     },
#     show_stdev=True,
#     apply_bonferroni= apply_correction_factor,
#     filename = "wilcoxon_freeze_rq3_with_ts2vec"
    
# )


# summary_df_freeze_rq3_with_ts2vec= summarize_backbone_performance(df_freeze_rq3_with_ts2vec)
# display(summary_df_freeze_rq3_with_ts2vec)





In [ ]:
import numpy as np
# FULL FINETUNE with TS2VEC for available techniques
dot,df_finetune_rq3_ts2vec = show_precedence_graph(
    df,
    variants_variables=["backbone","ft_strategy"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": [ "ResNet","RNN","Transformer","CNN"],   # Only use these backbones "TS2Vec",
        "select_tsk_pretext": ["TNC","Diet","TFC","LFR"],
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_finetune_rq3_with_ts2vec"
    
)

summary_df_finetune_rq3_ts2vec= summarize_backbone_performance(df_finetune_rq3_ts2vec)
display(summary_df_finetune_rq3_ts2vec)

# FREEZE with TS2VEC for available techniques

dot,df_freeze_rq3_with_ts2vec = show_precedence_graph(
    df,
    variants_variables=["backbone","ft_strategy"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Freeze"],    # Only use the full finetuning strategy
        # "select_backbones": [ "ResNet","RNN","Transformer","CNN"],   # Only use these backbones "TS2Vec",
        "select_tsk_pretext": ["TNC","Diet","TFC","LFR"],
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_freeze_rq3_with_ts2vec"
    
)


summary_df_freeze_rq3_with_ts2vec= summarize_backbone_performance(df_freeze_rq3_with_ts2vec)
display(summary_df_freeze_rq3_with_ts2vec)


In [ ]:
import numpy as np

# FULL FINETUNE with TS2VEC - one test per tsk_pretext
for pretext_task in ["Supervised", "TNC", "Diet", "TFC", "LFR"]:
    print(f"\n{'='*60}")
    print(f"FULL FINETUNE - {pretext_task}")
    print(f"{'='*60}")
    
    dot, df_finetune_rq3_ts2vec = show_precedence_graph(
        df,
        variants_variables=["backbone","ft_strategy"],
        filters={
            "select_ft_strategy": ["Full Finetune"],
            "select_tsk_pretext": [pretext_task],
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
        filename=f"wilcoxon_finetune_{pretext_task.lower()}"
    )
    
    summary_df_finetune_rq3_ts2vec = summarize_backbone_performance(df_finetune_rq3_ts2vec)
    display(summary_df_finetune_rq3_ts2vec)

# FREEZE with TS2VEC - one test per tsk_pretext  
for pretext_task in ["TNC", "Diet", "TFC", "LFR"]:
    print(f"\n{'='*60}")
    print(f"FREEZE - {pretext_task}")
    print(f"{'='*60}")
    
    dot, df_freeze_rq3_with_ts2vec = show_precedence_graph(
        df,
        variants_variables=["backbone","ft_strategy"],
        filters={
            "select_ft_strategy": ["Freeze"],
            "select_tsk_pretext": [pretext_task],
        },
        show_stdev=True,
        apply_bonferroni=apply_correction_factor,
        filename=f"wilcoxon_freeze_{pretext_task.lower()}"
    )
    
    summary_df_freeze_rq3_with_ts2vec = summarize_backbone_performance(df_freeze_rq3_with_ts2vec)
    display(summary_df_freeze_rq3_with_ts2vec)
